# Long Context Reorder — Lost-in-the-Middle 해결

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- **Lost-in-the-Middle** 문제가 발생하는 원인과 영향 이해하기
- `LongContextReorder`로 문서 순서를 재배치하는 방법 익히기
- 많은 문서를 검색해야 할 때 이 기법을 적용하는 시점 판단하기

## 사전 지식

- LLM이 긴 입력에서 중간 부분 정보를 상대적으로 덜 활용한다는 연구 결과
- VectorStoreRetriever의 기본 사용법

---

```mermaid
flowchart TD
    R[검색된 10개 문서<br/>유사도 순]:::input --> O[원래 순서]:::process
    O --> P[LongContextReorder]:::process
    P --> N[재배치 순서]:::output
    N --> M1[중요 문서<br/>처음과 끝에 배치]:::output
    N --> M2[중간 문서<br/>가운데 배치]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

## Lost-in-the-Middle 문제

연구 결과에 따르면 LLM은 긴 컨텍스트의 **중간 부분** 정보를 잘 활용하지 못해요:

```
문서 1 (처음) ← 잘 활용됨
문서 2
문서 3 (중간) ← 잘 활용 안 됨
문서 4
문서 5 (끝)   ← 잘 활용됨
```

## 해결책: LongContextReorder

중요한 문서를 처음과 끝에 배치하여 LLM이 더 잘 활용하도록 합니다.

> **실무 팁**: `k=10` 이상 많은 문서를 검색할 때 이 기법을 적용하면 효과적이에요. 검색 문서가 3~4개뿐이라면 효과가 크지 않습니다.

In [ ]:
from dotenv import load_dotenv
load_dotenv()


In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ---------------------------------------------------
# k=10 검색을 위한 VectorStore 준비
# ---------------------------------------------------

# ============================================================
# TODO: ai-story.txt를 로드하고 FAISS Retriever를 k=10으로 생성하세요
# 힌트: search_kwargs={"k": 10} — 많은 문서를 검색해야 재정렬 효과가 두드러짐
# 예상 결과: Retriever 준비 완료 메시지 출력
# ============================================================

# 문서 준비

# Retriever 생성 (많은 문서 검색)
# k=10: 충분히 많은 문서를 검색해야 재정렬 효과 확인 가능


In [ ]:
from langchain_community.document_transformers import LongContextReorder

# 1단계: 검색


# 2단계: LongContextReorder 적용
# LongContextReorder: 가장 관련성 높은 문서를 양 끝(1번, 마지막)에 배치
# 💡 팁: 이 변환은 문서 내용을 바꾸지 않고 순서만 바꿉니다

# 아래에 코드를 작성하세요


---

## 마무리 정리

### 핵심 요약

| 항목 | 내용 |
|------|------|
| 해결하는 문제 | Lost-in-the-Middle — LLM이 긴 컨텍스트에서 중간 문서를 놓치는 현상 |
| 재배치 원리 | 가장 관련도 낮은 문서를 중앙에, 가장 높은 문서를 양 끝에 배치 |
| 사용 방법 | `LongContextReorder().transform_documents(docs)` |
| 효과 | Retriever 성능은 그대로, LLM 최종 답변 품질이 향상 |
| 주의 | 재배치 자체는 검색 결과를 바꾸지 않음 — 반드시 상위 Retriever가 좋은 문서를 먼저 가져와야 함 |

### 언제 사용할까요?

- 검색 결과를 LLM에 전달하기 전 마지막 후처리 단계로 항상 적용해도 무방해요
- 특히 `top_k`가 6개 이상일 때 효과가 두드러져요
- 단독으로 쓰기보다 다른 Retriever와 조합해서 사용하는 것이 일반적이에요

### 다음 노트북 예고

**05-ParentDocumentRetriever** — 문서를 작은 청크로 분할해 검색하되, 실제 반환은 원본 부모 문서를 돌려주는 전략을 배워요. 검색 정밀도와 컨텍스트 풍부함을 동시에 잡는 방법을 살펴볼게요.